In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D
import pandas as pd
import os
import time   

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    GlobalMaxPooling2D,
    GlobalAveragePooling2D,
    Concatenate,
    Input
)
from tensorflow.keras.models import Model

# =========================================================
# ✅ EuPea Attention Module
# =========================================================
class EuPea(tf.keras.layers.Layer):

    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):

        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        # Euclidean distance normalization
        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)

        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)

        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        # Pearson-like normalization
        z = tf.square((x - x_mean) / (x_std + self.epsilon))

        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)

        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        # Attention map
        attention = tf.nn.sigmoid(d_norm + z_norm)

        return x * attention


# =========================================================
# ✅ Build ResNet50 + EuPea Feature Extractor
# =========================================================
def build_eupea_resnet50_feature_model():

    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    input_tensor = Input(shape=(224, 224, 3))

    x = tf.keras.applications.resnet50.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,
            base_model.get_layer("conv2_block3_out").output,
            base_model.get_layer("conv3_block4_out").output,
            base_model.get_layer("conv4_block6_out").output,
            base_model.get_layer("conv5_block3_out").output
        ]
    )

    features = backbone(x)

    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []

    for feat, pool in zip(features, pooling_layers):

        attn_feat = attention(feat)

        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)

    model = Model(inputs=input_tensor, outputs=final_feature)

    return model


# =========================================================
# ✅ Instantiate Feature Extractor
# =========================================================
eupea_feature_model = build_eupea_resnet50_feature_model()

print(eupea_feature_model.summary())


# =========================================================
# ✅ Dataset Directories
# =========================================================
base_dir = r"/kaggle/input/datasets/saurabhsainiji/anno-casewise-train-balanced-test-untouched/Anno-casewise-spilit-train-balanced-test-untouched_/Annocerv-balanced-311 4/Annocerv-balanced-311/Patient-wise/Split-3"

train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")

labels = ['Category-1', 'Category-2', 'Category-3']


# =========================================================
# ✅ Function to Check Corrupted Images
# =========================================================
def is_valid_image(img_path):

    try:
        img = Image.open(img_path)
        img.verify()
        return True

    except Exception:
        return False


# =========================================================
# ✅ Feature Extraction Function
# =========================================================
def extract_features_from_directory(data_dir):

    df = pd.DataFrame(columns=['features', 'labels', 'image_name'])

    corrupted_count = 0
    processed_count = 0

    for label_idx, label in enumerate(labels):

        label_dir = os.path.join(data_dir, label)

        if not os.path.exists(label_dir):

            print(f"⚠️ Warning: {label_dir} not found")
            continue

        print(f"\n📂 Processing Class: {label}")

        for subdir, dirs, files in os.walk(label_dir):

            for file in files:

                # Only image files
                if not (
                    file.lower().endswith(".jpg") or
                    file.lower().endswith(".jpeg") or
                    file.lower().endswith(".png")
                ):
                    continue

                img_path = os.path.join(subdir, file)

                # =====================================================
                # ✅ Skip corrupted images
                # =====================================================
                if not is_valid_image(img_path):

                    #print(f"❌ Corrupted Image Skipped: {img_path}")

                    corrupted_count += 1
                    continue

                try:

                    # Load image
                    img = tf.keras.preprocessing.image.load_img(
                        img_path,
                        target_size=(224, 224)
                    )

                    # Convert to array
                    x = tf.keras.preprocessing.image.img_to_array(img)

                    # Add batch dimension
                    x = tf.expand_dims(x, axis=0)

                    # =================================================
                    # ✅ Feature Extraction
                    # =================================================
                    features = eupea_feature_model(x)

                    features_np = features.numpy().flatten()

                    # Save features
                    df.loc[len(df)] = [
                        features_np,
                        label_idx,
                        file
                    ]

                    processed_count += 1

                    if processed_count % 50 == 0:
                        print(f"✅ Processed Images: {processed_count}")

                except Exception as e:

                    #print(f"❌ Error Processing: {img_path}")
                    #print(f"Error: {e}")

                    corrupted_count += 1
                    continue

    print("\n======================================")
    print(f"✅ Total Processed Images : {processed_count}")
    print(f"❌ Total Corrupted Images: {corrupted_count}")
    print("======================================")

    return df


# =========================================================
# ✅ TRAIN FEATURE EXTRACTION
# =========================================================
print("\n🚀 Extracting TRAIN Features...\n")

start_train_time = time.time()

res_50_train_features_df = extract_features_from_directory(train_dir)

end_train_time = time.time()

train_feature_time = end_train_time - start_train_time


# =========================================================
# ✅ TEST FEATURE EXTRACTION
# =========================================================
print("\n🚀 Extracting TEST Features...\n")

start_test_time = time.time()

res_50_test_features_df = extract_features_from_directory(test_dir)

end_test_time = time.time()

test_feature_time = end_test_time - start_test_time


# =========================================================
# ✅ TOTAL TIME
# =========================================================
total_time = train_feature_time + test_feature_time


# =========================================================
# ✅ DISPLAY SUMMARY
# =========================================================
print("\n======================================")
print("✅ Feature Extraction Summary")
print("======================================")

print("Training Features Shape : ", res_50_train_features_df.shape)
print("Testing Features Shape  : ", res_50_test_features_df.shape)

print(
    "Unique Training Feature Length:",
    res_50_train_features_df['features'].apply(len).unique()
)

print(
    "Unique Testing Feature Length:",
    res_50_test_features_df['features'].apply(len).unique()
)

print("\n======================================")
print("⏱️ Time Consumption")
print("======================================")

print(f"Training Time : {train_feature_time:.2f} seconds")
print(f"Testing Time  : {test_feature_time:.2f} seconds")
print(f"Total Time    : {total_time:.2f} seconds")


# =========================================================
# ✅ SAVE FEATURES
# =========================================================
res_50_train_features_df.to_pickle(
    "eupea_resnet50_train_features.pkl"
)

res_50_test_features_df.to_pickle(
    "eupea_resnet50_test_features.pkl"
)

print("\n✅ Feature files saved successfully!")

In [ ]:
print("BF-AD-AP-Resnet50 dataframe", res_50_train_features_df)

In [ ]:
import numpy as np
import joblib   # ✅ ADD THIS
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score

# --- ✅ Separate features, labels, and image names for training and testing sets ---
X_train = np.stack(res_50_train_features_df['features'].values)
y_train = np.array(res_50_train_features_df['labels'].values)

X_test = np.stack(res_50_test_features_df['features'].values)
y_test = np.array(res_50_test_features_df['labels'].values)
image_names = np.array(res_50_test_features_df['image_name'].values)

print("ResNet50 features with SVM (Train/Test evaluation)")



# =========================================================
# ✅ Measure TRAINING time
# =========================================================
start_train_time = time.time()

# --- ✅ Initialize and Train SVM ---
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time
# =========================================================

# ✅✅✅ SAVE MODEL TO KAGGLE OUTPUT DIRECTORY ✅✅✅
model_path = "/kaggle/working/svm_resnet50_fold3.pkl"
joblib.dump(svm, model_path)
print(f"✅ Trained SVM model saved at: {model_path}")

# =========================================================
# ✅ TRAINING predictions & metrics
# =========================================================
y_train_pred = svm.predict(X_train)
train_conf_matrix = confusion_matrix(y_train, y_train_pred)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='macro', zero_division=0)

train_sensitivity = np.mean([
    train_conf_matrix[i, i] / train_conf_matrix[i, :].sum()
    if train_conf_matrix[i, :].sum() > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_specificity = np.mean([
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[i, :]) - np.sum(train_conf_matrix[:, i]) + train_conf_matrix[i, i]) /
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[:, i]))
    if np.sum(train_conf_matrix[:, i]) > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_f1 = f1_score(y_train, y_train_pred, average='macro', zero_division=0)

# --- ✅ Print TRAINING results ---
print("\nTraining Performance:")
print(f"Accuracy: {train_accuracy:.6f}")
print(f"Specificity: {train_specificity:.6f}")
print(f"Sensitivity (Recall): {train_sensitivity:.6f}")
print(f"Precision: {train_precision:.6f}")
print(f"F1 Score: {train_f1:.6f}")
print(f"Training Time (seconds): {training_time:.4f}")
# =========================================================

# =========================================================
# ✅ Measure TESTING (inference) time
# =========================================================
start_test_time = time.time()
y_pred = svm.predict(X_test)
end_test_time = time.time()
testing_time = end_test_time - start_test_time
# =========================================================

# --- ✅ Compute confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

# --- ✅ Compute TEST metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum() if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])
specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) - np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i])) if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# --- ✅ Print TEST results ---
print("\nTest Performance:")
print(f"Accuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")
print(f"Testing Time (seconds): {testing_time:.4f}")

# --- ✅ Identify and print misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.layers import (
    GlobalMaxPooling2D,
    GlobalAveragePooling2D,
    Concatenate,
    Input
)
from tensorflow.keras.models import Model

# =========================================================
# ✅ EuPea Attention Module
# =========================================================
class EuPea(tf.keras.layers.Layer):

    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):

        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        # Euclidean distance normalization
        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)

        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)

        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        # Pearson-like normalization
        z = tf.square((x - x_mean) / (x_std + self.epsilon))

        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)

        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        # Attention map
        attention = tf.nn.sigmoid(d_norm + z_norm)

        return x * attention


# =========================================================
# ✅ Build ResNet101 + EuPea Feature Extractor
# =========================================================
def build_eupea_resnet101_feature_model():

    base_model = ResNet101(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    input_tensor = Input(shape=(224, 224, 3))

    x = tf.keras.applications.resnet.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,
            base_model.get_layer("conv2_block3_out").output,
            base_model.get_layer("conv3_block4_out").output,
            base_model.get_layer("conv4_block23_out").output,
            base_model.get_layer("conv5_block3_out").output
        ]
    )

    features = backbone(x)

    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []

    for feat, pool in zip(features, pooling_layers):

        attn_feat = attention(feat)

        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)

    model = Model(inputs=input_tensor, outputs=final_feature)

    return model


# =========================================================
# ✅ Instantiate Feature Extractor
# =========================================================
eupea_feature_model = build_eupea_resnet101_feature_model()

print(eupea_feature_model.summary())


# =========================================================
# ✅ Dataset Directories
# =========================================================
base_dir = r"/kaggle/input/datasets/saurabhsainiji/anno-casewise-train-balanced-test-untouched/Anno-casewise-spilit-train-balanced-test-untouched_/Annocerv-balanced-311 4/Annocerv-balanced-311/Patient-wise/Split-3"

train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")

labels = ['Category-1', 'Category-2', 'Category-3']


# =========================================================
# ✅ Function to Check Corrupted Images
# =========================================================
def is_valid_image(img_path):

    try:
        img = Image.open(img_path)
        img.verify()
        return True

    except Exception:
        return False


# =========================================================
# ✅ Feature Extraction Function
# =========================================================
def extract_features_from_directory(data_dir):

    df = pd.DataFrame(columns=['features', 'labels', 'image_name'])

    corrupted_count = 0
    processed_count = 0

    for label_idx, label in enumerate(labels):

        label_dir = os.path.join(data_dir, label)

        if not os.path.exists(label_dir):

            print(f"⚠️ Warning: {label_dir} not found")
            continue

        print(f"\n📂 Processing Class: {label}")

        for subdir, dirs, files in os.walk(label_dir):

            for file in files:

                # Only image files
                if not (
                    file.lower().endswith(".jpg") or
                    file.lower().endswith(".jpeg") or
                    file.lower().endswith(".png")
                ):
                    continue

                img_path = os.path.join(subdir, file)

                # =====================================================
                # ✅ Skip corrupted images
                # =====================================================
                if not is_valid_image(img_path):

                    #print(f"❌ Corrupted Image Skipped: {img_path}")

                    corrupted_count += 1
                    continue

                try:

                    # Load image
                    img = tf.keras.preprocessing.image.load_img(
                        img_path,
                        target_size=(224, 224)
                    )

                    # Convert image to array
                    x = tf.keras.preprocessing.image.img_to_array(img)

                    # Add batch dimension
                    x = tf.expand_dims(x, axis=0)

                    # =================================================
                    # ✅ Feature Extraction
                    # =================================================
                    features = eupea_feature_model(x)

                    features_np = features.numpy().flatten()

                    # Save features
                    df.loc[len(df)] = [
                        features_np,
                        label_idx,
                        file
                    ]

                    processed_count += 1

                    if processed_count % 50 == 0:
                        print(f"✅ Processed Images: {processed_count}")

                except Exception as e:

                    #print(f"❌ Error Processing: {img_path}")
                    #print(f"Error: {e}")

                    corrupted_count += 1
                    continue

    print("\n======================================")
    print(f"✅ Total Processed Images : {processed_count}")
    print(f"❌ Total Corrupted Images: {corrupted_count}")
    print("======================================")

    return df


# =========================================================
# ✅ TRAIN FEATURE EXTRACTION
# =========================================================
print("\n🚀 Extracting TRAIN Features...\n")

start_train_time = time.time()

res_101_train_features_df = extract_features_from_directory(train_dir)

end_train_time = time.time()

train_feature_time = end_train_time - start_train_time


# =========================================================
# ✅ TEST FEATURE EXTRACTION
# =========================================================
print("\n🚀 Extracting TEST Features...\n")

start_test_time = time.time()

res_101_test_features_df = extract_features_from_directory(test_dir)

end_test_time = time.time()

test_feature_time = end_test_time - start_test_time


# =========================================================
# ✅ TOTAL TIME
# =========================================================
total_time = train_feature_time + test_feature_time


# =========================================================
# ✅ DISPLAY SUMMARY
# =========================================================
print("\n======================================")
print("✅ Feature Extraction Summary")
print("======================================")

print("Training Features Shape : ", res_101_train_features_df.shape)
print("Testing Features Shape  : ", res_101_test_features_df.shape)

print(
    "Unique Training Feature Length:",
    res_101_train_features_df['features'].apply(len).unique()
)

print(
    "Unique Testing Feature Length:",
    res_101_test_features_df['features'].apply(len).unique()
)

print("\n======================================")
print("⏱️ Time Consumption")
print("======================================")

print(f"Training Time : {train_feature_time:.2f} seconds")
print(f"Testing Time  : {test_feature_time:.2f} seconds")
print(f"Total Time    : {total_time:.2f} seconds")


# =========================================================
# ✅ SAVE FEATURES
# =========================================================
res_101_train_features_df.to_pickle(
    "eupea_resnet101_train_features.pkl"
)

res_101_test_features_df.to_pickle(
    "eupea_resnet101_test_features.pkl"
)

print("\n✅ Feature files saved successfully!")

In [ ]:
import numpy as np
import joblib   # ✅ ADD THIS
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score

# --- ✅ Separate features, labels, and image names for training and testing sets ---
X_train = np.stack(res_101_train_features_df['features'].values)
y_train = np.array(res_101_train_features_df['labels'].values)

X_test = np.stack(res_101_test_features_df['features'].values)
y_test = np.array(res_101_test_features_df['labels'].values)
image_names = np.array(res_101_test_features_df['image_name'].values)

print("ResNet101 features with SVM (Train/Test evaluation)")


# =========================================================
# ✅ Measure TRAINING time
# =========================================================
start_train_time = time.time()

# --- ✅ Initialize and Train SVM ---
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time
# =========================================================

# ✅✅✅ SAVE MODEL TO KAGGLE OUTPUT DIRECTORY ✅✅✅
model_path = "/kaggle/working/svm_resnet101_fold3.pkl"
joblib.dump(svm, model_path)
print(f"✅ Trained SVM model saved at: {model_path}")

# =========================================================
# ✅ TRAINING predictions & metrics
# =========================================================
y_train_pred = svm.predict(X_train)
train_conf_matrix = confusion_matrix(y_train, y_train_pred)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='macro', zero_division=0)

train_sensitivity = np.mean([
    train_conf_matrix[i, i] / train_conf_matrix[i, :].sum()
    if train_conf_matrix[i, :].sum() > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_specificity = np.mean([
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[i, :]) - np.sum(train_conf_matrix[:, i]) + train_conf_matrix[i, i]) /
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[:, i]))
    if np.sum(train_conf_matrix[:, i]) > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_f1 = f1_score(y_train, y_train_pred, average='macro', zero_division=0)

# --- ✅ Print TRAINING results ---
print("\nTraining Performance:")
print(f"Accuracy: {train_accuracy:.6f}")
print(f"Specificity: {train_specificity:.6f}")
print(f"Sensitivity (Recall): {train_sensitivity:.6f}")
print(f"Precision: {train_precision:.6f}")
print(f"F1 Score: {train_f1:.6f}")
print(f"Training Time (seconds): {training_time:.4f}")
# =========================================================

# =========================================================
# ✅ Measure TESTING (inference) time
# =========================================================
start_test_time = time.time()
y_pred = svm.predict(X_test)
end_test_time = time.time()
testing_time = end_test_time - start_test_time
# =========================================================

# --- ✅ Compute confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

# --- ✅ Compute TEST metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum() if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])
specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) - np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i])) if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# --- ✅ Print TEST results ---
print("\nTest Performance:")
print(f"Accuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")
print(f"Testing Time (seconds): {testing_time:.4f}")

# --- ✅ Identify and print misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.layers import (
    GlobalMaxPooling2D,
    GlobalAveragePooling2D,
    Concatenate,
    Input
)
from tensorflow.keras.models import Model

# =========================================================
# ✅ EuPea Attention Module
# =========================================================
class EuPea(tf.keras.layers.Layer):

    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):

        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        # Euclidean distance normalization
        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)

        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)

        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        # Pearson-like normalization
        z = tf.square((x - x_mean) / (x_std + self.epsilon))

        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)

        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        # Attention map
        attention = tf.nn.sigmoid(d_norm + z_norm)

        return x * attention


# =========================================================
# ✅ Build ResNet152 + EuPea Feature Extractor
# =========================================================
def build_eupea_resnet152_feature_model():

    base_model = ResNet152(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    input_tensor = Input(shape=(224, 224, 3))

    x = tf.keras.applications.resnet.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,
            base_model.get_layer("conv2_block3_out").output,
            base_model.get_layer("conv3_block8_out").output,
            base_model.get_layer("conv4_block36_out").output,
            base_model.get_layer("conv5_block3_out").output
        ]
    )

    features = backbone(x)

    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []

    for feat, pool in zip(features, pooling_layers):

        attn_feat = attention(feat)

        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)

    model = Model(inputs=input_tensor, outputs=final_feature)

    return model


# =========================================================
# ✅ Instantiate Feature Extractor
# =========================================================
eupea_feature_model = build_eupea_resnet152_feature_model()

print(eupea_feature_model.summary())


# =========================================================
# ✅ Dataset Directories
# =========================================================
base_dir = r"/kaggle/input/datasets/saurabhsainiji/anno-casewise-train-balanced-test-untouched/Anno-casewise-spilit-train-balanced-test-untouched_/Annocerv-balanced-311 4/Annocerv-balanced-311/Patient-wise/Split-3"

train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")

labels = ['Category-1', 'Category-2', 'Category-3']


# =========================================================
# ✅ Function to Check Corrupted Images
# =========================================================
def is_valid_image(img_path):

    try:
        img = Image.open(img_path)
        img.verify()
        return True

    except Exception:
        return False


# =========================================================
# ✅ Feature Extraction Function
# =========================================================
def extract_features_from_directory(data_dir):

    df = pd.DataFrame(columns=['features', 'labels', 'image_name'])

    corrupted_count = 0
    processed_count = 0

    for label_idx, label in enumerate(labels):

        label_dir = os.path.join(data_dir, label)

        if not os.path.exists(label_dir):

            print(f"⚠️ Warning: {label_dir} not found")
            continue

        print(f"\n📂 Processing Class: {label}")

        for subdir, dirs, files in os.walk(label_dir):

            for file in files:

                # Only image files
                if not (
                    file.lower().endswith(".jpg") or
                    file.lower().endswith(".jpeg") or
                    file.lower().endswith(".png")
                ):
                    continue

                img_path = os.path.join(subdir, file)

                # =====================================================
                # ✅ Skip corrupted images
                # =====================================================
                if not is_valid_image(img_path):

                    #print(f"❌ Corrupted Image Skipped: {img_path}")

                    corrupted_count += 1
                    continue

                try:

                    # Load image
                    img = tf.keras.preprocessing.image.load_img(
                        img_path,
                        target_size=(224, 224)
                    )

                    # Convert image to array
                    x = tf.keras.preprocessing.image.img_to_array(img)

                    # Add batch dimension
                    x = tf.expand_dims(x, axis=0)

                    # =================================================
                    # ✅ Feature Extraction
                    # =================================================
                    features = eupea_feature_model(x)

                    features_np = features.numpy().flatten()

                    # Save features
                    df.loc[len(df)] = [
                        features_np,
                        label_idx,
                        file
                    ]

                    processed_count += 1

                    if processed_count % 50 == 0:
                        print(f"✅ Processed Images: {processed_count}")

                except Exception as e:

                    #print(f"❌ Error Processing: {img_path}")
                    #print(f"Error: {e}")

                    corrupted_count += 1
                    continue

    print("\n======================================")
    print(f"✅ Total Processed Images : {processed_count}")
    print(f"❌ Total Corrupted Images: {corrupted_count}")
    print("======================================")

    return df


# =========================================================
# ✅ TRAIN FEATURE EXTRACTION
# =========================================================
print("\n🚀 Extracting TRAIN Features...\n")

start_train_time = time.time()

res_152_train_features_df = extract_features_from_directory(train_dir)

end_train_time = time.time()

train_feature_time = end_train_time - start_train_time


# =========================================================
# ✅ TEST FEATURE EXTRACTION
# =========================================================
print("\n🚀 Extracting TEST Features...\n")

start_test_time = time.time()

res_152_test_features_df = extract_features_from_directory(test_dir)

end_test_time = time.time()

test_feature_time = end_test_time - start_test_time


# =========================================================
# ✅ TOTAL TIME
# =========================================================
total_time = train_feature_time + test_feature_time


# =========================================================
# ✅ DISPLAY SUMMARY
# =========================================================
print("\n======================================")
print("✅ Feature Extraction Summary (ResNet152 + EuPea)")
print("======================================")

print("Training Features Shape : ", res_152_train_features_df.shape)
print("Testing Features Shape  : ", res_152_test_features_df.shape)

print(
    "Unique Training Feature Length:",
    res_152_train_features_df['features'].apply(len).unique()
)

print(
    "Unique Testing Feature Length:",
    res_152_test_features_df['features'].apply(len).unique()
)

print("\n======================================")
print("⏱️ Time Consumption")
print("======================================")

print(f"Training Time : {train_feature_time:.2f} seconds")
print(f"Testing Time  : {test_feature_time:.2f} seconds")
print(f"Total Time    : {total_time:.2f} seconds")


# =========================================================
# ✅ SAVE FEATURES
# =========================================================
res_152_train_features_df.to_pickle(
    "eupea_resnet152_train_features.pkl"
)

res_152_test_features_df.to_pickle(
    "eupea_resnet152_test_features.pkl"
)

print("\n✅ Feature files saved successfully!")

In [ ]:
import numpy as np
import joblib   # ✅ ADD THIS
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score

# --- ✅ Separate features, labels, and image names for training and testing sets ---
X_train = np.stack(res_152_train_features_df['features'].values)
y_train = np.array(res_152_train_features_df['labels'].values)

X_test = np.stack(res_152_test_features_df['features'].values)
y_test = np.array(res_152_test_features_df['labels'].values)
image_names = np.array(res_152_test_features_df['image_name'].values)

print("ResNet152 features with SVM (Train/Test evaluation)")


# =========================================================
# ✅ Measure TRAINING time
# =========================================================
start_train_time = time.time()

# --- ✅ Initialize and Train SVM ---
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time
# =========================================================

# ✅✅✅ SAVE MODEL TO KAGGLE OUTPUT DIRECTORY ✅✅✅
model_path = "/kaggle/working/svm_resnet152_fold3.pkl"
joblib.dump(svm, model_path)
print(f"✅ Trained SVM model saved at: {model_path}")

# =========================================================
# ✅ TRAINING predictions & metrics
# =========================================================
y_train_pred = svm.predict(X_train)
train_conf_matrix = confusion_matrix(y_train, y_train_pred)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='macro', zero_division=0)

train_sensitivity = np.mean([
    train_conf_matrix[i, i] / train_conf_matrix[i, :].sum()
    if train_conf_matrix[i, :].sum() > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_specificity = np.mean([
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[i, :]) - np.sum(train_conf_matrix[:, i]) + train_conf_matrix[i, i]) /
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[:, i]))
    if np.sum(train_conf_matrix[:, i]) > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_f1 = f1_score(y_train, y_train_pred, average='macro', zero_division=0)

# --- ✅ Print TRAINING results ---
print("\nTraining Performance:")
print(f"Accuracy: {train_accuracy:.6f}")
print(f"Specificity: {train_specificity:.6f}")
print(f"Sensitivity (Recall): {train_sensitivity:.6f}")
print(f"Precision: {train_precision:.6f}")
print(f"F1 Score: {train_f1:.6f}")
print(f"Training Time (seconds): {training_time:.4f}")
# =========================================================

# =========================================================
# ✅ Measure TESTING (inference) time
# =========================================================
start_test_time = time.time()
y_pred = svm.predict(X_test)
end_test_time = time.time()
testing_time = end_test_time - start_test_time
# =========================================================

# --- ✅ Compute confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

# --- ✅ Compute TEST metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum() if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])
specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) - np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i])) if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# --- ✅ Print TEST results ---
print("\nTest Performance:")
print(f"Accuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")
print(f"Testing Time (seconds): {testing_time:.4f}")

# --- ✅ Identify and print misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")


# Feature Fusion

In [ ]:
import numpy as np
import pandas as pd

# --- ✅ Function to concatenate feature vectors ---
def concat_features(df1, df2, df3):
    combined_features = []
    for f1, f2, f3 in zip(df1['features'], df2['features'], df3['features']):
        combined = np.concatenate([f1, f2, f3])
        combined_features.append(combined)
    return pd.DataFrame({
        'features': combined_features,
        'labels': df1['labels'],
        'image_name': df1['image_name']
    })

# --- ✅ Combine training and testing features ---
combined_train_df = concat_features(res_50_train_features_df, res_101_train_features_df, res_152_train_features_df)
combined_test_df = concat_features(res_50_test_features_df, res_101_test_features_df, res_152_test_features_df)

# --- ✅ Print summary ---
print("✅ Combined training set shape:", combined_train_df.shape)
print("✅ Combined testing set shape:", combined_test_df.shape)
print("Feature vector length after concatenation:", len(combined_train_df['features'][0]))

# --- ✅ Optional: Save if you want later ---
# combined_train_df.to_pickle("combined_resnet_train_features.pkl")
# combined_test_df.to_pickle("combined_resnet_test_features.pkl")


In [ ]:
import numpy as np
import joblib
import time
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score
from sklearn.preprocessing import MinMaxScaler

# --- ✅ Separate features, labels, and image names ---
X_train = np.stack(combined_train_df['features'].values)
y_train = np.array(combined_train_df['labels'].values)

X_test = np.stack(combined_test_df['features'].values)
y_test = np.array(combined_test_df['labels'].values)
image_names = np.array(combined_test_df['image_name'].values)

print("Combined ResNet50+101+152 features with SVM (Train/Test evaluation)")

# =========================================================
# ✅ Feature Scaling (CORRECTED)
# =========================================================
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # ✅ IMPORTANT: transform, not fit_transform
# =========================================================

# =========================================================
# ✅ Measure TRAINING time
# =========================================================
start_train_time = time.time()

svm = SVC(kernel='linear')
svm.fit(X_train_scaled, y_train)   # ✅ FIXED

end_train_time = time.time()
training_time = end_train_time - start_train_time
# =========================================================

# --- ✅ Save SVM model ---
model_path = "/kaggle/working/svm_combined_resnet50_101_152_fold3.pkl"
joblib.dump(svm, model_path)
print("✅ SVM model saved at:", model_path)

# =========================================================
# ✅ TRAINING performance
# =========================================================
y_train_pred = svm.predict(X_train_scaled)   # ✅ FIXED
train_conf_matrix = confusion_matrix(y_train, y_train_pred)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='macro', zero_division=0)

train_sensitivity = np.mean([
    train_conf_matrix[i, i] / train_conf_matrix[i, :].sum()
    if train_conf_matrix[i, :].sum() > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_specificity = np.mean([
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[i, :]) -
     np.sum(train_conf_matrix[:, i]) + train_conf_matrix[i, i]) /
    (np.sum(train_conf_matrix) - np.sum(train_conf_matrix[:, i]))
    if np.sum(train_conf_matrix[:, i]) > 0 else 0
    for i in range(train_conf_matrix.shape[0])
])

train_f1 = f1_score(y_train, y_train_pred, average='macro', zero_division=0)

print("\nTraining Performance:")
print(f"Accuracy: {train_accuracy:.6f}")
print(f"Specificity: {train_specificity:.6f}")
print(f"Sensitivity (Recall): {train_sensitivity:.6f}")
print(f"Precision: {train_precision:.6f}")
print(f"F1 Score: {train_f1:.6f}")
print(f"Training Time (seconds): {training_time:.4f}")

# =========================================================
# ✅ Measure TESTING (Inference) time
# =========================================================
start_test_time = time.time()
y_pred = svm.predict(X_test_scaled)   # ✅ FIXED
end_test_time = time.time()
testing_time = end_test_time - start_test_time
# =========================================================

# --- ✅ Confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

# --- ✅ TEST metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)

sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum()
    if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])

specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) -
     np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i]))
    if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])

f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

print("\nTest Performance:")
print(f"Accuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")
print(f"Testing Time (seconds): {testing_time:.4f}")

# --- ✅ Misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")
